# 06 — Hybrid InternImage+VMamba + Rabi + Adam

**Research Objective:** Evaluate feature-level fusion of InternImage and VMamba on the Rabi crop classification dataset.

**Fusion:** Feature concatenation (not logit fusion). InternImage (768) + VMamba (384) -> concat (1152) -> Linear(576) -> Linear(576) -> Dropout -> Classifier.

> **Instructions:** Set `SMOKE_TEST = False` for full training. On GPU: set `VMAMBA_FORWARD_TYPE = 'v2'`.

In [ ]:
import os, sys, math, time, pickle, warnings, json
import copy
from pathlib import Path
from collections import OrderedDict
from functools import partial
from typing import Optional, Any

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.utils.checkpoint as checkpoint
from torch.utils.data import Dataset, DataLoader

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    confusion_matrix, classification_report
)
from sklearn.utils.class_weight import compute_class_weight
from timm.models.layers import DropPath, trunc_normal_

In [ ]:
# ─────────────────────────────────────────────────────────────
# 1. EXPERIMENT CONFIGURATION
# ─────────────────────────────────────────────────────────────
# GPU notes:
#   - Change VMAMBA_FORWARD_TYPE to 'v2' on GPU for full CUDA selective scan
#   - DCNv3_pytorch (pure PyTorch) is used; for max perf compile CUDA ext
VMAMBA_FORWARD_TYPE = 'v05_noz'

SMOKE_TEST    = True
SMOKE_BATCHES = 2
SMOKE_EPOCHS  = 2

BATCH_SIZE   = 128
MAX_EPOCHS   = 100
LR           = 1e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP    = 1.0
DROPOUT_RATE = 0.3
SEED         = 42
NUM_WORKERS  = 4

# Scheduler
SCHED_FACTOR   = 0.5
SCHED_PATIENCE = 8
SCHED_MIN_LR   = 1e-6

# Early stopping
ES_PATIENCE = 15

# Mixed precision
USE_AMP = True

torch.manual_seed(SEED); np.random.seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.deterministic = False   # False required for benchmark mode

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {DEVICE}")
if DEVICE.type == 'cuda':
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 2. DATASET LOADING AND AUDIT
# Pickle format: dict{class_name: [{"data": ndarray(12,7,7), ...}]}
# ─────────────────────────────────────────────────────────────
NB_DIR = Path('.').resolve()    # Models_DL/
PICKLE_PATH = NB_DIR / 'FINAL_RABI.pickle'
assert PICKLE_PATH.exists(), f"Dataset not found: {PICKLE_PATH}"

print(f"Loading {PICKLE_PATH.name} (FINAL_RABI.pickle) ...")
t0 = time.time()
with open(PICKLE_PATH, 'rb') as f:
    raw = pickle.load(f)
print(f"  Loaded in {time.time()-t0:.1f}s | type={type(raw).__name__} | keys={{'...' if len(raw)>5 else list(raw.keys())}}")

classes = sorted(raw.keys())
class_to_id = {c: i for i, c in enumerate(classes)}
id_to_class = {i: c for c, i in class_to_id.items()}
NUM_CLASSES = len(classes)
print(f"  Classes ({NUM_CLASSES}): {classes}")

arrays, labels = [], []
for c in classes:
    for s in raw[c]:
        if isinstance(s, dict) and 'data' in s:
            arrays.append(s['data'])  # ndarray (12, 7, 7) float64
            labels.append(c)
        else:
            arrays.append(s)
            labels.append(c)
del raw

total = len(arrays)
print(f"  Total samples: {total:,}")
print(f"  Sample shape:  {arrays[0].shape}, dtype={arrays[0].dtype}")

# NaN / Inf check (sample 1000 random)
check_idx = np.random.choice(total, min(1000, total), replace=False)
nan_count = sum(np.any(np.isnan(arrays[i])) for i in check_idx)
inf_count = sum(np.any(np.isinf(arrays[i])) for i in check_idx)
print(f"  NaN check (1000 samples): {nan_count} affected | Inf: {inf_count} affected")

# Per-class counts
print("  Per-class sample counts:")
label_arr = np.array(labels)
for c in classes:
    cnt = int((label_arr == c).sum())
    print(f"    {c:30s}: {cnt:6,}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 3. SPLIT LOADING, NORMALIZATION, DATASET CLASS, DATALOADERS
# ─────────────────────────────────────────────────────────────
# Frozen splits are stored inside Models_DL/SPLITS/
# (80/10/10 stratified, seed=42 — generated once and locked)
SPLITS_DIR = NB_DIR / 'SPLITS'

if SPLITS_DIR.exists() and (SPLITS_DIR / 'rabi_train_idx.npy').exists():
    print(f"Loading frozen splits from {SPLITS_DIR.resolve()} ...")
    train_idx = np.load(SPLITS_DIR / 'rabi_train_idx.npy')
    val_idx   = np.load(SPLITS_DIR / 'rabi_val_idx.npy')
    test_idx  = np.load(SPLITS_DIR / 'rabi_test_idx.npy')
    print(f"  Loaded: train={len(train_idx):,} / val={len(val_idx):,} / test={len(test_idx):,}")
else:
    print("WARNING: SPLITS/ folder or index files not found in Models_DL/.")
    print("         Please copy the 6 .npy split files to Models_DL/SPLITS/")
    print("         Expected files:")
    for fn in ['rabi_train_idx.npy', 'rabi_val_idx.npy', 'rabi_test_idx.npy']:
        print(f"           SPLITS/{fn}")
    print()
    print("Generating deterministic fallback splits (80/10/10, seed=42).")
    rng = np.random.RandomState(42)
    label_arr_local = np.array(labels)
    tr, va, te = [], [], []
    for c in classes:
        ci = np.where(label_arr_local == c)[0]; rng.shuffle(ci)
        n = len(ci); nt = int(round(n*0.80)); nv = int(round(n*0.10))
        tr.extend(ci[:nt]); va.extend(ci[nt:nt+nv]); te.extend(ci[nt+nv:])
    train_idx = np.array(tr, dtype=np.int64)
    val_idx   = np.array(va, dtype=np.int64)
    test_idx  = np.array(te, dtype=np.int64)
    SPLITS_DIR.mkdir(parents=True, exist_ok=True)
    np.save(SPLITS_DIR / 'rabi_train_idx.npy', train_idx)
    np.save(SPLITS_DIR / 'rabi_val_idx.npy', val_idx)
    np.save(SPLITS_DIR / 'rabi_test_idx.npy', test_idx)
    print(f"  Saved splits to {SPLITS_DIR}")
    print(f"  Generated: train={len(train_idx):,} / val={len(val_idx):,} / test={len(test_idx):,}")


# Verify zero overlap
assert len(set(train_idx) & set(val_idx)) == 0, "TRAIN/VAL OVERLAP!"
assert len(set(train_idx) & set(test_idx)) == 0, "TRAIN/TEST OVERLAP!"
assert len(set(val_idx) & set(test_idx)) == 0, "VAL/TEST OVERLAP!"
print("  Split overlap check: PASSED")

# Normalization: per-channel z-score from TRAINING data ONLY
print("Computing train-only normalization stats ...")
train_arr = np.stack([arrays[i] for i in train_idx])  # (N_train, 12, 7, 7)
norm_mean = np.mean(train_arr, axis=(0, 2, 3))         # (12,)
norm_std  = np.std(train_arr, axis=(0, 2, 3))          # (12,)
norm_std  = np.where(norm_std < 1e-6, 1.0, norm_std)
del train_arr
print(f"  mean: {norm_mean.round(4)}")
print(f"  std:  {norm_std.round(4)}")


class CropDataset(Dataset):
    '''
    Satellite tile dataset with optional spatial augmentation.
    Valid augmentations for 7x7 satellite patches (scientifically justified):
      - hflip, vflip: fields are spatially symmetric (no preferred orientation)
      - rot90: satellite view from above — rotation invariant
    NOT applied: color jitter, blur, erase — would corrupt spectral bands.
    augment=True only for training set.
    '''
    def __init__(self, indices, augment=False):
        self.indices = indices
        self.augment = augment
        self.mean_t = torch.tensor(norm_mean, dtype=torch.float32).view(12, 1, 1)
        self.std_t  = torch.tensor(norm_std,  dtype=torch.float32).view(12, 1, 1)
    def __len__(self): return len(self.indices)
    def __getitem__(self, idx):
        real_idx = self.indices[idx]
        tensor = torch.tensor(arrays[real_idx], dtype=torch.float32)  # (12, 7, 7)
        tensor = (tensor - self.mean_t) / (self.std_t + 1e-7)
        if self.augment:
            if torch.rand(1).item() > 0.5:
                tensor = torch.flip(tensor, dims=[2])           # hflip
            if torch.rand(1).item() > 0.5:
                tensor = torch.flip(tensor, dims=[1])           # vflip
            k = torch.randint(0, 4, (1,)).item()
            if k > 0:
                tensor = torch.rot90(tensor, k=k, dims=[1, 2])  # rot90
        label  = class_to_id[labels[real_idx]]
        return tensor, torch.tensor(label, dtype=torch.long)

BS = BATCH_SIZE if not SMOKE_TEST else 4
train_loader = DataLoader(CropDataset(train_idx, augment=(not SMOKE_TEST)), batch_size=BS, shuffle=True,  num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'), drop_last=True)
val_loader   = DataLoader(CropDataset(val_idx),   batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
test_loader  = DataLoader(CropDataset(test_idx),  batch_size=BS, shuffle=False, num_workers=NUM_WORKERS, pin_memory=(DEVICE.type=='cuda'))
print(f"DataLoaders ready: batch_size={BS}, num_workers={NUM_WORKERS}")
xb, yb = next(iter(train_loader))
print(f"  Batch shape: x={xb.shape}, y={yb.shape}, y_range=[{yb.min().item()}, {yb.max().item()}]")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 4. MODEL ARCHITECTURE — Hybrid InternImage + VMamba
# Feature-level fusion. NOT logit concatenation.
# All classes inline. No external project imports.
# ─────────────────────────────────────────────────────────────
# ════════════════════════════════════════════════════════════════
# VMamba Cross-Scan (pure PyTorch, extracted from csm_triton.py)
# Source: official VMamba repo / classification / models / csm_triton.py
# No CUDA extension required.
# ════════════════════════════════════════════════════════════════
def _cross_scan_fwd(x: torch.Tensor, in_channel_first=True, out_channel_first=True, scans=0):
    if in_channel_first:
        B, C, H, W = x.shape
        if scans == 0:
            y = x.new_empty((B, 4, C, H * W))
            y[:, 0, :, :] = x.flatten(2, 3)
            y[:, 1, :, :] = x.transpose(dim0=2, dim1=3).flatten(2, 3)
            y[:, 2:4, :, :] = torch.flip(y[:, 0:2, :, :], dims=[-1])
        elif scans == 1:
            y = x.view(B, 1, C, H * W).repeat(1, 4, 1, 1)
        elif scans == 2:
            y = x.view(B, 1, C, H * W).repeat(1, 2, 1, 1)
            y = torch.cat([y, y.flip(dims=[-1])], dim=1)
    else:
        B, H, W, C = x.shape
        if scans == 0:
            y = x.new_empty((B, H * W, 4, C))
            y[:, :, 0, :] = x.flatten(1, 2)
            y[:, :, 1, :] = x.transpose(dim0=1, dim1=2).flatten(1, 2)
            y[:, :, 2:4, :] = torch.flip(y[:, :, 0:2, :], dims=[1])
        elif scans == 1:
            y = x.view(B, H * W, 1, C).repeat(1, 1, 4, 1)
        elif scans == 2:
            y = x.view(B, H * W, 1, C).repeat(1, 1, 2, 1)
            y = torch.cat([y, y.flip(dims=[1])], dim=2)
    if in_channel_first and (not out_channel_first):
        y = y.permute(0, 3, 1, 2).contiguous()
    elif (not in_channel_first) and out_channel_first:
        y = y.permute(0, 2, 3, 1).contiguous()
    return y


def _cross_merge_fwd(y: torch.Tensor, in_channel_first=True, out_channel_first=True, scans=0):
    if out_channel_first:
        B, K, D, H, W = y.shape
        y = y.view(B, K, D, -1)
        if scans == 0:
            y = y[:, 0:2] + y[:, 2:4].flip(dims=[-1]).view(B, 2, D, -1)
            y = y[:, 0] + y[:, 1].view(B, -1, W, H).transpose(dim0=2, dim1=3).contiguous().view(B, D, -1)
        elif scans == 1:
            y = y.sum(1)
        elif scans == 2:
            y = y[:, 0:2] + y[:, 2:4].flip(dims=[-1]).view(B, 2, D, -1)
            y = y.sum(1)
    else:
        B, H, W, K, D = y.shape
        y = y.view(B, -1, K, D)
        if scans == 0:
            y = y[:, :, 0:2] + y[:, :, 2:4].flip(dims=[1]).view(B, -1, 2, D)
            y = y[:, :, 0] + y[:, :, 1].view(B, W, H, -1).transpose(dim0=1, dim1=2).contiguous().view(B, -1, D)
        elif scans == 1:
            y = y.sum(2)
        elif scans == 2:
            y = y[:, :, 0:2] + y[:, :, 2:4].flip(dims=[1]).view(B, -1, 2, D)
            y = y.sum(2)
    if in_channel_first and (not out_channel_first):
        y = y.permute(0, 2, 1).contiguous()
    elif (not in_channel_first) and out_channel_first:
        y = y.permute(0, 2, 1).contiguous()
    return y


class CrossScanF(torch.autograd.Function):
    @staticmethod
    def forward(ctx, x, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0):
        ctx.in_channel_first = in_channel_first
        ctx.out_channel_first = out_channel_first
        ctx.one_by_one = one_by_one
        ctx.scans = scans
        if one_by_one:
            B, K, C, H, W = x.shape
            if not in_channel_first:
                B, H, W, K, C = x.shape
        else:
            B, C, H, W = x.shape
            if not in_channel_first:
                B, H, W, C = x.shape
        ctx.shape = (B, C, H, W)
        return _cross_scan_fwd(x, in_channel_first=in_channel_first, out_channel_first=out_channel_first, scans=scans)

    @staticmethod
    def backward(ctx, y):
        in_cf = ctx.in_channel_first; out_cf = ctx.out_channel_first; scans = ctx.scans
        B, C, H, W = ctx.shape
        # y has shape (B, 4, C, H*W) if out_cf else (B, H*W, 4, C)
        # reshape to 5D for cross_merge_fwd which expects (B, K, D, H, W)
        if out_cf:
            y = y.view(B, -1, C, H, W)
        else:
            y = y.view(B, H, W, -1, C)
        grad = _cross_merge_fwd(y, in_channel_first=out_cf, out_channel_first=in_cf, scans=scans)
        # reshape back to original input shape
        if in_cf:
            grad = grad.view(B, -1, H, W)
        else:
            grad = grad.view(B, H, W, -1)
        return grad, None, None, None, None


class CrossMergeF(torch.autograd.Function):
    @staticmethod
    def forward(ctx, y, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0):
        ctx.in_channel_first = in_channel_first
        ctx.out_channel_first = out_channel_first
        ctx.one_by_one = one_by_one
        ctx.scans = scans
        B, K, D, H, W = y.shape
        if not out_channel_first:
            B, H, W, K, D = y.shape
        ctx.shape = (B, D, H, W)
        return _cross_merge_fwd(y, in_channel_first=in_channel_first, out_channel_first=out_channel_first, scans=scans)

    @staticmethod
    def backward(ctx, x):
        in_cf = ctx.in_channel_first; out_cf = ctx.out_channel_first; scans = ctx.scans
        B, D, H, W = ctx.shape
        # x has shape (B, D, H*W) if in_cf else (B, H*W, D)
        # reshape to 4D for cross_scan_fwd which expects (B, D, H, W)
        if in_cf:
            x = x.view(B, D, H, W)
        else:
            x = x.view(B, H, W, D)
        grad = _cross_scan_fwd(x, in_channel_first=out_cf, out_channel_first=in_cf, scans=scans)
        # reshape to 5D (B, 4, D, H, W) or (B, H, W, 4, D)
        if out_cf:
            grad = grad.view(B, 4, D, H, W)
        else:
            grad = grad.view(B, H, W, 4, D)
        return grad, None, None, None, None


def cross_scan_fn(x, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0, force_torch=False):
    return CrossScanF.apply(x, in_channel_first, out_channel_first, one_by_one, scans)


def cross_merge_fn(y, in_channel_first=True, out_channel_first=True, one_by_one=False, scans=0, force_torch=False):
    return CrossMergeF.apply(y, in_channel_first, out_channel_first, one_by_one, scans)


# ════════════════════════════════════════════════════════════════
# VMamba Selective Scan (pure PyTorch fallback, from csms6s.py)
# Source: official VMamba repo / classification / models / csms6s.py
# This is the reference implementation used when no CUDA extension
# is compiled (backend="torch"). Functionally identical.
# ════════════════════════════════════════════════════════════════
def selective_scan_fn(
    u, delta, A, B, C, D=None, delta_bias=None,
    delta_softplus=True, oflex=True, backend=None,
):
    """Pure-PyTorch selective scan (O(L*N) memory, used on CPU/GPU without extensions)."""
    dtype_in = u.dtype
    Batch, K, N, L = B.shape
    KCdim = u.shape[1]
    Cdim = int(KCdim / K)
    if delta_bias is not None:
        delta = delta + delta_bias[..., None]
    if delta_softplus:
        delta = torch.nn.functional.softplus(delta)
    u, delta, A, B, C = u.float(), delta.float(), A.float(), B.float(), C.float()
    B = B.view(Batch, K, 1, N, L).repeat(1, 1, Cdim, 1, 1).view(Batch, KCdim, N, L)
    C = C.view(Batch, K, 1, N, L).repeat(1, 1, Cdim, 1, 1).view(Batch, KCdim, N, L)
    deltaA = torch.exp(torch.einsum('bdl,dn->bdln', delta, A))
    deltaB_u = torch.einsum('bdl,bdnl,bdl->bdln', delta, B, u)
    x = A.new_zeros((Batch, KCdim, N))
    ys = []
    for i in range(L):
        x = deltaA[:, :, i, :] * x + deltaB_u[:, :, i, :]
        y = torch.einsum('bdn,bdn->bd', x, C[:, :, :, i])
        ys.append(y)
    y = torch.stack(ys, dim=2)
    out = y if D is None else y + u * D.unsqueeze(-1)
    return out if oflex else out.to(dtype=dtype_in)


# ════════════════════════════════════════════════════════════════
# VMamba Model Classes
# Source: official VMamba repo / classification / models / vmamba.py
# All classes needed for VSSM with forward_type="v05_noz"
# (pure PyTorch, no CUDA extension required)
# ════════════════════════════════════════════════════════════════

class Linear2d(nn.Linear):
    def forward(self, x):
        return F.conv2d(x, self.weight[:, :, None, None], self.bias)
    def _load_from_state_dict(self, state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs):
        state_dict[prefix + "weight"] = state_dict[prefix + "weight"].view(self.weight.shape)
        return super()._load_from_state_dict(state_dict, prefix, local_metadata, strict, missing_keys, unexpected_keys, error_msgs)


class LayerNorm2d(nn.LayerNorm):
    def forward(self, x):
        x = x.permute(0, 2, 3, 1)
        x = nn.functional.layer_norm(x, self.normalized_shape, self.weight, self.bias, self.eps)
        return x.permute(0, 3, 1, 2)


class Permute(nn.Module):
    def __init__(self, *args):
        super().__init__()
        self.args = args
    def forward(self, x):
        return x.permute(*self.args)


class SoftmaxSpatial(nn.Softmax):
    def forward(self, x):
        if self.dim == -1:
            B, C, H, W = x.shape
            return super().forward(x.view(B, C, -1)).view(B, C, H, W)
        elif self.dim == 1:
            B, H, W, C = x.shape
            return super().forward(x.view(B, -1, C)).view(B, H, W, C)
        raise NotImplementedError


class Mlp(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer=nn.GELU, drop=0., channels_first=False):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        Linear = Linear2d if channels_first else nn.Linear
        self.fc1 = Linear(in_features, hidden_features)
        self.act = act_layer()
        self.fc2 = Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class mamba_init:
    @staticmethod
    def dt_init(dt_rank, d_inner, dt_scale=1.0, dt_init="random", dt_min=0.001, dt_max=0.1, dt_init_floor=1e-4):
        dt_proj = nn.Linear(dt_rank, d_inner, bias=True)
        dt_init_std = dt_rank ** -0.5 * dt_scale
        if dt_init == "random":
            nn.init.uniform_(dt_proj.weight, -dt_init_std, dt_init_std)
        else:
            nn.init.constant_(dt_proj.weight, dt_init_std)
        dt = torch.exp(torch.rand(d_inner) * (math.log(dt_max) - math.log(dt_min)) + math.log(dt_min)).clamp(min=dt_init_floor)
        inv_dt = dt + torch.log(-torch.expm1(-dt))
        with torch.no_grad():
            dt_proj.bias.copy_(inv_dt)
        return dt_proj

    @staticmethod
    def A_log_init(d_state, d_inner, copies=-1, device=None, merge=True):
        A = torch.arange(1, d_state + 1, dtype=torch.float32, device=device).view(1, -1).repeat(d_inner, 1).contiguous()
        A_log = torch.log(A)
        if copies > 0:
            A_log = A_log[None].repeat(copies, 1, 1).contiguous()
            if merge:
                A_log = A_log.flatten(0, 1)
        A_log = nn.Parameter(A_log)
        A_log._no_weight_decay = True
        return A_log

    @staticmethod
    def D_init(d_inner, copies=-1, device=None, merge=True):
        D = torch.ones(d_inner, device=device)
        if copies > 0:
            D = D[None].repeat(copies, 1).contiguous()
            if merge:
                D = D.flatten(0, 1)
        D = nn.Parameter(D)
        D._no_weight_decay = True
        return D

    @classmethod
    def init_dt_A_D(cls, d_state, dt_rank, d_inner, dt_scale, dt_init, dt_min, dt_max, dt_init_floor, k_group=4):
        dt_projs = [cls.dt_init(dt_rank, d_inner, dt_scale, dt_init, dt_min, dt_max, dt_init_floor) for _ in range(k_group)]
        dt_projs_weight = nn.Parameter(torch.stack([t.weight for t in dt_projs], dim=0))
        dt_projs_bias = nn.Parameter(torch.stack([t.bias for t in dt_projs], dim=0))
        del dt_projs
        A_logs = cls.A_log_init(d_state, d_inner, copies=k_group, merge=True)
        Ds = cls.D_init(d_inner, copies=k_group, merge=True)
        return A_logs, Ds, dt_projs_weight, dt_projs_bias


class SS2D(nn.Module):
    """
    VMamba State Space 2D module.
    Using forward_type='v05_noz': pure PyTorch selective scan, no z-gate, no einsum.
    """
    def __init__(
        self, d_model=96, d_state=16, ssm_ratio=2.0, dt_rank="auto",
        act_layer=nn.SiLU, d_conv=3, conv_bias=True,
        dropout=0.0, bias=False,
        dt_min=0.001, dt_max=0.1, dt_init="random", dt_scale=1.0, dt_init_floor=1e-4,
        initialize="v0", forward_type="v05_noz", channel_first=False, **kwargs,
    ):
        super().__init__()
        self.k_group = 4
        self.d_model = int(d_model)
        self.d_state = int(d_state)
        self.d_inner = int(ssm_ratio * d_model)
        self.dt_rank = int(math.ceil(self.d_model / 16) if dt_rank == "auto" else dt_rank)
        self.channel_first = channel_first
        self.with_dconv = d_conv > 1
        Linear = Linear2d if channel_first else nn.Linear
        self.forward = self.forwardv2

        def checkpostfix(tag, value):
            ret = value[-len(tag):] == tag
            if ret: value = value[:-len(tag)]
            return ret, value

        self.disable_force32, forward_type = checkpostfix("_no32", forward_type)
        self.oact, forward_type = checkpostfix("_oact", forward_type)
        self.disable_z, forward_type = checkpostfix("_noz", forward_type)
        self.disable_z_act, forward_type = checkpostfix("_nozact", forward_type)

        # out_norm
        LayerNorm = LayerNorm2d if channel_first else nn.LayerNorm
        self.out_norm = LayerNorm(self.d_inner)

        # forward core dispatcher
        no_einsum = "v05" in forward_type or "v04" in forward_type
        force_fp32 = not self.disable_force32
        self.forward_core = partial(
            self._forward_core,
            force_fp32=force_fp32,
            no_einsum=no_einsum,
            scans=0,
        )

        # projections
        d_proj = self.d_inner if self.disable_z else (self.d_inner * 2)
        self.in_proj = Linear(self.d_model, d_proj, bias=bias)
        self.act = act_layer()

        if self.with_dconv:
            self.conv2d = nn.Conv2d(
                in_channels=self.d_inner, out_channels=self.d_inner,
                groups=self.d_inner, bias=conv_bias,
                kernel_size=d_conv, padding=(d_conv - 1) // 2,
            )

        self.x_proj = [nn.Linear(self.d_inner, (self.dt_rank + self.d_state * 2), bias=False) for _ in range(self.k_group)]
        self.x_proj_weight = nn.Parameter(torch.stack([t.weight for t in self.x_proj], dim=0))
        del self.x_proj

        self.out_act = nn.GELU() if self.oact else nn.Identity()
        self.out_proj = Linear(self.d_inner, self.d_model, bias=bias)
        self.dropout = nn.Dropout(dropout) if dropout > 0. else nn.Identity()

        if initialize == "v0":
            self.A_logs, self.Ds, self.dt_projs_weight, self.dt_projs_bias = mamba_init.init_dt_A_D(
                self.d_state, self.dt_rank, self.d_inner, dt_scale, dt_init, dt_min, dt_max, dt_init_floor, k_group=self.k_group,
            )

    def _forward_core(self, x, force_fp32=False, no_einsum=False, scans=0):
        B, D, H, W = x.shape
        N = self.d_state
        K, D, R = self.k_group, self.d_inner, self.dt_rank
        L = H * W
        to_fp32 = lambda *args: (_a.to(torch.float32) for _a in args)

        xs = cross_scan_fn(x, in_channel_first=True, out_channel_first=True, scans=scans)

        if no_einsum:
            x_dbl = F.conv1d(xs.view(B, -1, L), self.x_proj_weight.view(-1, D, 1), groups=K)
            dts, Bs, Cs = torch.split(x_dbl.view(B, K, -1, L), [R, N, N], dim=2)
            dts = F.conv1d(dts.contiguous().view(B, -1, L), self.dt_projs_weight.view(K * D, -1, 1), groups=K)
        else:
            x_dbl = torch.einsum("b k d l, k c d -> b k c l", xs, self.x_proj_weight)
            dts, Bs, Cs = torch.split(x_dbl, [R, N, N], dim=2)
            dts = torch.einsum("b k r l, k d r -> b k d l", dts, self.dt_projs_weight)

        xs  = xs.view(B, -1, L)
        dts = dts.contiguous().view(B, -1, L)
        As  = -self.A_logs.to(torch.float).exp()
        Ds  = self.Ds.to(torch.float)
        Bs  = Bs.contiguous().view(B, K, N, L)
        Cs  = Cs.contiguous().view(B, K, N, L)
        delta_bias = self.dt_projs_bias.view(-1).to(torch.float)

        if force_fp32:
            xs, dts, Bs, Cs = to_fp32(xs, dts, Bs, Cs)

        ys = selective_scan_fn(xs, dts, As, Bs, Cs, Ds, delta_bias, delta_softplus=True).view(B, K, -1, H, W)
        y  = cross_merge_fn(ys, in_channel_first=True, out_channel_first=True, scans=scans)
        y  = y.view(B, -1, H, W)
        if not self.channel_first:
            y = y.view(B, -1, H * W).transpose(1, 2).contiguous().view(B, H, W, -1)
        y = self.out_norm(y)
        return y.to(x.dtype)

    def forwardv2(self, x, **kwargs):
        x = self.in_proj(x)
        if not self.disable_z:
            x, z = x.chunk(2, dim=(1 if self.channel_first else -1))
            if not self.disable_z_act:
                z = self.act(z)
        if not self.channel_first:
            x = x.permute(0, 3, 1, 2).contiguous()
        if self.with_dconv:
            x = self.conv2d(x)
        x = self.act(x)
        y = self.forward_core(x)
        y = self.out_act(y)
        if not self.disable_z:
            y = y * z
        return self.dropout(self.out_proj(y))


class VSSBlock(nn.Module):
    def __init__(
        self, hidden_dim=0, drop_path=0., norm_layer=nn.LayerNorm, channel_first=False,
        ssm_d_state=16, ssm_ratio=2.0, ssm_dt_rank="auto", ssm_act_layer=nn.SiLU,
        ssm_conv=3, ssm_conv_bias=True, ssm_drop_rate=0., ssm_init="v0",
        forward_type="v2", mlp_ratio=4.0, mlp_act_layer=nn.GELU, mlp_drop_rate=0.,
        gmlp=False, use_checkpoint=False, post_norm=False, _SS2D=None, **kwargs,
    ):
        super().__init__()
        if _SS2D is None: _SS2D = SS2D
        self.ssm_branch = ssm_ratio > 0
        self.mlp_branch = mlp_ratio > 0
        self.use_checkpoint = use_checkpoint
        self.post_norm = post_norm
        if self.ssm_branch:
            self.norm = norm_layer(hidden_dim)
            self.op = _SS2D(
                d_model=hidden_dim, d_state=ssm_d_state, ssm_ratio=ssm_ratio,
                dt_rank=ssm_dt_rank, act_layer=ssm_act_layer,
                d_conv=ssm_conv, conv_bias=ssm_conv_bias,
                dropout=ssm_drop_rate, initialize=ssm_init,
                forward_type=forward_type, channel_first=channel_first,
            )
        self.drop_path = DropPath(drop_path)
        if self.mlp_branch:
            self.norm2 = norm_layer(hidden_dim)
            self.mlp = Mlp(in_features=hidden_dim, hidden_features=int(hidden_dim * mlp_ratio), act_layer=mlp_act_layer, drop=mlp_drop_rate, channels_first=channel_first)

    def _forward(self, x):
        if self.ssm_branch:
            x = x + self.drop_path(self.norm(self.op(x)) if self.post_norm else self.op(self.norm(x)))
        if self.mlp_branch:
            x = x + self.drop_path(self.norm2(self.mlp(x)) if self.post_norm else self.mlp(self.norm2(x)))
        return x

    def forward(self, x):
        return checkpoint.checkpoint(self._forward, x) if self.use_checkpoint else self._forward(x)


class VSSM(nn.Module):
    """
    Visual State Space Model — VMamba backbone.
    Source: official VMamba repo / classification / models / vmamba.py
    Uses forward_type='v05_noz' (pure PyTorch, no z-gate, no einsum).
    On GPU server: change to forward_type='v2' for full CUDA performance.
    """
    def __init__(
        self, patch_size=4, in_chans=3, num_classes=1000,
        depths=[2, 2, 9, 2], dims=[96, 192, 384, 768],
        ssm_d_state=16, ssm_ratio=2.0, ssm_dt_rank="auto",
        ssm_act_layer="silu", ssm_conv=3, ssm_conv_bias=True,
        ssm_drop_rate=0.0, ssm_init="v0", forward_type="v2",
        mlp_ratio=4.0, mlp_act_layer="gelu", mlp_drop_rate=0.0, gmlp=False,
        drop_path_rate=0.1, patch_norm=True, norm_layer="ln",
        downsample_version="v2", patchembed_version="v1",
        use_checkpoint=False, posembed=False, imgsize=224,
        _SS2D=SS2D, **kwargs,
    ):
        super().__init__()
        self.channel_first = norm_layer.lower() in ["bn", "ln2d"]
        self.num_classes = num_classes
        self.num_layers = len(depths)
        if isinstance(dims, int):
            dims = [int(dims * 2 ** i) for i in range(self.num_layers)]
        self.num_features = dims[-1]
        self.dims = dims
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(depths))]
        _NORMLAYERS = dict(ln=nn.LayerNorm, ln2d=LayerNorm2d, bn=nn.BatchNorm2d)
        _ACTLAYERS  = dict(silu=nn.SiLU, gelu=nn.GELU, relu=nn.ReLU, sigmoid=nn.Sigmoid)
        norm_layer_cls = _NORMLAYERS.get(norm_layer.lower(), nn.LayerNorm)
        ssm_act_layer_cls = _ACTLAYERS.get(ssm_act_layer.lower(), nn.SiLU)
        mlp_act_layer_cls = _ACTLAYERS.get(mlp_act_layer.lower(), nn.GELU)

        self.pos_embed = self._pos_embed(dims[0], patch_size, imgsize) if posembed else None

        if patchembed_version == "v1":
            self.patch_embed = nn.Sequential(
                nn.Conv2d(in_chans, dims[0], kernel_size=patch_size, stride=patch_size, bias=True),
                (nn.Identity() if self.channel_first else Permute(0, 2, 3, 1)),
                (norm_layer_cls(dims[0]) if patch_norm else nn.Identity()),
            )
        else:
            stride = patch_size // 2; ksize = stride + 1; pad = 1
            self.patch_embed = nn.Sequential(
                nn.Conv2d(in_chans, dims[0] // 2, kernel_size=ksize, stride=stride, padding=pad),
                (nn.Identity() if (self.channel_first or not patch_norm) else Permute(0, 2, 3, 1)),
                (norm_layer_cls(dims[0] // 2) if patch_norm else nn.Identity()),
                (nn.Identity() if (self.channel_first or not patch_norm) else Permute(0, 3, 1, 2)),
                nn.GELU(),
                nn.Conv2d(dims[0] // 2, dims[0], kernel_size=ksize, stride=stride, padding=pad),
                (nn.Identity() if self.channel_first else Permute(0, 2, 3, 1)),
                (norm_layer_cls(dims[0]) if patch_norm else nn.Identity()),
            )

        def _make_downsample(dim, out_dim, norm_layer_cls, channel_first):
            if downsample_version == "v3":
                return nn.Sequential(
                    (nn.Identity() if channel_first else Permute(0, 3, 1, 2)),
                    nn.Conv2d(dim, out_dim, kernel_size=3, stride=2, padding=1),
                    (nn.Identity() if channel_first else Permute(0, 2, 3, 1)),
                    norm_layer_cls(out_dim),
                )
            return nn.Sequential(
                (nn.Identity() if channel_first else Permute(0, 3, 1, 2)),
                nn.Conv2d(dim, out_dim, kernel_size=2, stride=2),
                (nn.Identity() if channel_first else Permute(0, 2, 3, 1)),
                norm_layer_cls(out_dim),
            )

        self.layers = nn.ModuleList()
        for i_layer in range(self.num_layers):
            downsample = _make_downsample(
                self.dims[i_layer], self.dims[i_layer + 1], norm_layer_cls, self.channel_first
            ) if (i_layer < self.num_layers - 1) else nn.Identity()

            blocks = [VSSBlock(
                hidden_dim=dims[i_layer], drop_path=dpr[sum(depths[:i_layer]) + d],
                norm_layer=norm_layer_cls, channel_first=self.channel_first,
                ssm_d_state=ssm_d_state, ssm_ratio=ssm_ratio, ssm_dt_rank=ssm_dt_rank,
                ssm_act_layer=ssm_act_layer_cls, ssm_conv=ssm_conv, ssm_conv_bias=ssm_conv_bias,
                ssm_drop_rate=ssm_drop_rate, ssm_init=ssm_init,
                forward_type=forward_type, mlp_ratio=mlp_ratio,
                mlp_act_layer=mlp_act_layer_cls, mlp_drop_rate=mlp_drop_rate, gmlp=gmlp,
                use_checkpoint=use_checkpoint, _SS2D=_SS2D,
            ) for d in range(depths[i_layer])]
            self.layers.append(nn.Sequential(OrderedDict(blocks=nn.Sequential(*blocks), downsample=downsample)))

        self.classifier = nn.Sequential(OrderedDict(
            norm=norm_layer_cls(self.num_features),
            permute=(Permute(0, 3, 1, 2) if not self.channel_first else nn.Identity()),
            avgpool=nn.AdaptiveAvgPool2d(1),
            flatten=nn.Flatten(1),
            head=nn.Linear(self.num_features, num_classes),
        ))
        self.apply(self._init_weights)

    @staticmethod
    def _pos_embed(embed_dims, patch_size, img_size):
        p = img_size // patch_size
        pos = nn.Parameter(torch.zeros(1, embed_dims, p, p))
        trunc_normal_(pos, std=0.02)
        return pos

    def _init_weights(self, m):
        if isinstance(m, nn.Linear):
            trunc_normal_(m.weight, std=.02)
            if m.bias is not None: nn.init.constant_(m.bias, 0)
        elif isinstance(m, nn.LayerNorm):
            nn.init.constant_(m.bias, 0); nn.init.constant_(m.weight, 1.0)

    def forward(self, x):
        x = self.patch_embed(x)
        if self.pos_embed is not None:
            pos = self.pos_embed.permute(0, 2, 3, 1) if not self.channel_first else self.pos_embed
            x = x + pos
        for layer in self.layers:
            x = layer(x)
        return self.classifier(x)


# ════════════════════════════════════════════════════════════════
# DCNv3 Pure-PyTorch Implementation
# Source: official InternImage repo / classification / ops_dcnv3 /
#   functions/dcnv3_func.py  (dcnv3_core_pytorch)
#   modules/dcnv3.py         (DCNv3_pytorch)
# No CUDA compilation required. On GPU server with compiled DCNv3,
# the CUDA version gives better throughput but is functionally identical.
# ════════════════════════════════════════════════════════════════

class _to_channels_first(nn.Module):
    def forward(self, x): return x.permute(0, 3, 1, 2)

class _to_channels_last(nn.Module):
    def forward(self, x): return x.permute(0, 2, 3, 1)

def _build_norm_layer(dim, norm_layer, in_format='channels_last', out_format='channels_last', eps=1e-6):
    layers = []
    if norm_layer == 'BN':
        if in_format == 'channels_last': layers.append(_to_channels_first())
        layers.append(nn.BatchNorm2d(dim))
        if out_format == 'channels_last': layers.append(_to_channels_last())
    elif norm_layer == 'LN':
        if in_format == 'channels_first': layers.append(_to_channels_last())
        layers.append(nn.LayerNorm(dim, eps=eps))
        if out_format == 'channels_first': layers.append(_to_channels_first())
    return nn.Sequential(*layers)

def _build_act_layer(act_layer):
    if act_layer == 'ReLU': return nn.ReLU(inplace=True)
    if act_layer == 'SiLU': return nn.SiLU(inplace=True)
    if act_layer == 'GELU': return nn.GELU()
    raise NotImplementedError(f'build_act_layer does not support {act_layer}')


def _get_reference_points(spatial_shapes, device, kernel_h, kernel_w, dilation_h, dilation_w, pad_h=0, pad_w=0, stride_h=1, stride_w=1):
    _, H_, W_, _ = spatial_shapes
    H_out = (H_ - (dilation_h * (kernel_h - 1) + 1)) // stride_h + 1
    W_out = (W_ - (dilation_w * (kernel_w - 1) + 1)) // stride_w + 1
    ref_y, ref_x = torch.meshgrid(
        torch.linspace((dilation_h*(kernel_h-1))//2+0.5, (dilation_h*(kernel_h-1))//2+0.5+(H_out-1)*stride_h, H_out, dtype=torch.float32, device=device),
        torch.linspace((dilation_w*(kernel_w-1))//2+0.5, (dilation_w*(kernel_w-1))//2+0.5+(W_out-1)*stride_w, W_out, dtype=torch.float32, device=device),
    )
    ref_y = ref_y.reshape(-1)[None] / H_
    ref_x = ref_x.reshape(-1)[None] / W_
    return torch.stack((ref_x, ref_y), -1).reshape(1, H_out, W_out, 1, 2)


def _generate_dilation_grids(spatial_shapes, kernel_h, kernel_w, dilation_h, dilation_w, group, device):
    _, H_, W_, _ = spatial_shapes
    x, y = torch.meshgrid(
        torch.linspace(-((dilation_w*(kernel_w-1))//2), -((dilation_w*(kernel_w-1))//2)+(kernel_w-1)*dilation_w, kernel_w, dtype=torch.float32, device=device),
        torch.linspace(-((dilation_h*(kernel_h-1))//2), -((dilation_h*(kernel_h-1))//2)+(kernel_h-1)*dilation_h, kernel_h, dtype=torch.float32, device=device),
    )
    grid = torch.stack([x/W_, y/H_], -1).reshape(-1, 1, 2).repeat(1, group, 1).permute(1, 0, 2)
    return grid.reshape(1, 1, 1, group * kernel_h * kernel_w, 2)


def dcnv3_core_pytorch(input, offset, mask, kernel_h, kernel_w, stride_h, stride_w, pad_h, pad_w, dilation_h, dilation_w, group, group_channels, offset_scale, remove_center=False):
    input = F.pad(input, [0, 0, pad_h, pad_h, pad_w, pad_w])
    N_, H_in, W_in, _ = input.shape
    _, H_out, W_out, _ = offset.shape
    ref  = _get_reference_points(input.shape, input.device, kernel_h, kernel_w, dilation_h, dilation_w, pad_h, pad_w, stride_h, stride_w)
    grid = _generate_dilation_grids(input.shape, kernel_h, kernel_w, dilation_h, dilation_w, group, input.device)
    spatial_norm = torch.tensor([W_in, H_in]).reshape(1,1,1,2).repeat(1,1,1,group*kernel_h*kernel_w).to(input.device)
    P_ = kernel_h * kernel_w
    sampling_locations = (ref + grid * offset_scale).repeat(N_, 1, 1, 1, 1).flatten(3, 4)
    sampling_locations = sampling_locations + offset * offset_scale / spatial_norm
    sampling_grids = 2 * sampling_locations - 1
    input_ = input.view(N_, H_in*W_in, group*group_channels).transpose(1,2).reshape(N_*group, group_channels, H_in, W_in)
    sampling_grid_ = sampling_grids.view(N_, H_out*W_out, group, P_, 2).transpose(1,2).flatten(0,1)
    sampling_input_ = F.grid_sample(input_, sampling_grid_, mode='bilinear', padding_mode='zeros', align_corners=False)
    mask = mask.view(N_, H_out*W_out, group, P_).transpose(1,2).reshape(N_*group, 1, H_out*W_out, P_)
    output = (sampling_input_ * mask).sum(-1).view(N_, group*group_channels, H_out*W_out)
    return output.transpose(1,2).reshape(N_, H_out, W_out, -1).contiguous()


class DCNv3_pytorch(nn.Module):
    """
    DCNv3 pure-PyTorch implementation.
    Source: official InternImage / ops_dcnv3 / modules / dcnv3.py
    Functionally identical to the CUDA-compiled version; no make.sh needed.
    """
    def __init__(self, channels=64, kernel_size=3, dw_kernel_size=None, stride=1, pad=1,
                 dilation=1, group=4, offset_scale=1.0, act_layer='GELU', norm_layer='LN',
                 center_feature_scale=False, remove_center=False, **kwargs):
        super().__init__()
        if channels % group != 0:
            raise ValueError(f'channels ({channels}) must be divisible by group ({group})')
        self.channels = channels
        self.kernel_size = kernel_size
        self.stride = stride
        self.dilation = dilation
        self.pad = pad
        self.group = group
        self.group_channels = channels // group
        self.offset_scale = offset_scale
        self.remove_center = int(remove_center)
        self.center_feature_scale = center_feature_scale
        if dw_kernel_size is not None:
            self.dw_conv = nn.Sequential(
                nn.Conv2d(channels, channels, kernel_size=dw_kernel_size, stride=1,
                          padding=(dw_kernel_size-1)//2, groups=channels),
                _build_norm_layer(channels, 'LN', in_format='channels_first', out_format='channels_last'),
                _build_act_layer(act_layer)
            )
        self.offset = nn.Linear(channels, group * kernel_size * kernel_size * 2)
        self.mask   = nn.Linear(channels, group * kernel_size * kernel_size)
        self.input_proj = nn.Linear(channels, channels)
        self.output_proj = nn.Linear(channels, channels)
        self._reset_parameters()
        if center_feature_scale:
            self.center_feature_scale_proj_weight = nn.Parameter(torch.zeros((group, channels), dtype=torch.float))
            self.center_feature_scale_proj_bias   = nn.Parameter(torch.tensor(0.0, dtype=torch.float).view((1,)).repeat(group))
            self.center_feature_scale_module = nn.Sequential(nn.Linear(channels, group))

    def _reset_parameters(self):
        nn.init.constant_(self.offset.weight.data, 0.)
        nn.init.constant_(self.offset.bias.data, 0.)
        nn.init.constant_(self.mask.weight.data, 0.)
        nn.init.constant_(self.mask.bias.data, 0.)
        nn.init.xavier_uniform_(self.input_proj.weight.data)
        nn.init.constant_(self.input_proj.bias.data, 0.)
        nn.init.xavier_uniform_(self.output_proj.weight.data)
        nn.init.constant_(self.output_proj.bias.data, 0.)

    def forward(self, input):
        N, H, W, _ = input.shape
        x = self.input_proj(input)
        dtype = x.dtype
        if hasattr(self, 'dw_conv'):
            x1 = x.permute(0,3,1,2)
            x1 = self.dw_conv(x1)
            offset = self.offset(x1)
            mask   = self.mask(x1).reshape(N,H,W,self.group*self.kernel_size*self.kernel_size)
        else:
            offset = self.offset(x)
            mask   = self.mask(x).reshape(N,H,W,self.group*self.kernel_size*self.kernel_size)
        mask = F.softmax(mask, -1).reshape(N,H,W,self.group,self.kernel_size*self.kernel_size)
        x = dcnv3_core_pytorch(
            x, offset, mask,
            self.kernel_size, self.kernel_size,
            self.stride, self.stride,
            self.pad, self.pad,
            self.dilation, self.dilation,
            self.group, self.group_channels,
            self.offset_scale,
        )
        if self.center_feature_scale:
            center_feature_scale = self.center_feature_scale_module(F.linear(input, self.center_feature_scale_proj_weight, self.center_feature_scale_proj_bias)).sigmoid().view(N,H,W,self.group,1).repeat(1,1,1,1,self.group_channels).flatten(-2)
            x = x * (1 - center_feature_scale) + input.view(N,H,W,self.channels) * center_feature_scale
        x = self.output_proj(x)
        return x


# ════════════════════════════════════════════════════════════════
# InternImage Model Classes
# Source: official InternImage repo / classification / models / intern_image.py
# Classes needed: MLPLayer, InternImageLayer, InternImageBlock, DownsampleLayer
# ════════════════════════════════════════════════════════════════

class _MLPLayer(nn.Module):
    def __init__(self, in_features, hidden_features=None, out_features=None, act_layer='GELU', drop=0.):
        super().__init__()
        out_features = out_features or in_features
        hidden_features = hidden_features or in_features
        self.fc1  = nn.Linear(in_features, hidden_features)
        self.act  = _build_act_layer(act_layer)
        self.fc2  = nn.Linear(hidden_features, out_features)
        self.drop = nn.Dropout(drop)
    def forward(self, x):
        return self.drop(self.fc2(self.drop(self.act(self.fc1(x)))))


class _InternImageLayer(nn.Module):
    def __init__(self, core_op, channels, groups, mlp_ratio=4., drop=0., drop_path=0.,
                 act_layer='GELU', norm_layer='LN', post_norm=False, layer_scale=None,
                 offset_scale=1.0, with_cp=False, dw_kernel_size=None,
                 res_post_norm=False, center_feature_scale=False, remove_center=False):
        super().__init__()
        self.with_cp = with_cp
        self.post_norm = post_norm
        self.norm1 = _build_norm_layer(channels, 'LN')
        self.dcn   = core_op(channels=channels, kernel_size=3, stride=1, pad=1, dilation=1,
                             group=groups, offset_scale=offset_scale, act_layer=act_layer,
                             norm_layer=norm_layer, dw_kernel_size=dw_kernel_size,
                             center_feature_scale=center_feature_scale, remove_center=remove_center)
        self.drop_path = DropPath(drop_path) if drop_path > 0. else nn.Identity()
        self.norm2 = _build_norm_layer(channels, 'LN')
        self.mlp   = _MLPLayer(in_features=channels, hidden_features=int(channels*mlp_ratio), act_layer=act_layer, drop=drop)
        self.layer_scale = layer_scale is not None
        if self.layer_scale:
            self.gamma1 = nn.Parameter(layer_scale * torch.ones(channels), requires_grad=True)
            self.gamma2 = nn.Parameter(layer_scale * torch.ones(channels), requires_grad=True)

    def forward(self, x):
        def _inner(x):
            if not self.layer_scale:
                if self.post_norm:
                    x = x + self.drop_path(self.norm1(self.dcn(x)))
                    x = x + self.drop_path(self.norm2(self.mlp(x)))
                else:
                    x = x + self.drop_path(self.dcn(self.norm1(x)))
                    x = x + self.drop_path(self.mlp(self.norm2(x)))
            else:
                if self.post_norm:
                    x = x + self.drop_path(self.gamma1 * self.norm1(self.dcn(x)))
                    x = x + self.drop_path(self.gamma2 * self.norm2(self.mlp(x)))
                else:
                    x = x + self.drop_path(self.gamma1 * self.dcn(self.norm1(x)))
                    x = x + self.drop_path(self.gamma2 * self.mlp(self.norm2(x)))
            return x
        return checkpoint.checkpoint(_inner, x) if (self.with_cp and x.requires_grad) else _inner(x)


class _DownsampleLayer(nn.Module):
    def __init__(self, channels, norm_layer='LN'):
        super().__init__()
        self.conv = nn.Conv2d(channels, 2*channels, kernel_size=3, stride=2, padding=1, bias=False)
        self.norm = _build_norm_layer(2*channels, norm_layer, 'channels_first', 'channels_last')
    def forward(self, x):
        return self.norm(self.conv(x.permute(0,3,1,2)))


class _InternImageBlock(nn.Module):
    """
    One stage of InternImage = N InternImageLayer blocks + optional downsample.
    Source: official InternImage repo / classification / models / intern_image.py
    """
    def __init__(self, core_op, channels, depth, groups, downsample=True, mlp_ratio=4.,
                 drop=0., drop_path=0., act_layer='GELU', norm_layer='LN', post_norm=False,
                 offset_scale=1.0, layer_scale=None, with_cp=False, dw_kernel_size=None,
                 post_norm_block_ids=None, res_post_norm=False, center_feature_scale=False,
                 remove_center=False):
        super().__init__()
        self.post_norm = post_norm
        self.center_feature_scale = center_feature_scale
        self.blocks = nn.ModuleList([
            _InternImageLayer(
                core_op=core_op, channels=channels, groups=groups, mlp_ratio=mlp_ratio, drop=drop,
                drop_path=drop_path[i] if isinstance(drop_path, list) else drop_path,
                act_layer=act_layer, norm_layer=norm_layer, post_norm=post_norm,
                layer_scale=layer_scale, offset_scale=offset_scale, with_cp=with_cp,
                dw_kernel_size=dw_kernel_size, res_post_norm=res_post_norm,
                center_feature_scale=center_feature_scale, remove_center=remove_center,
            ) for i in range(depth)
        ])
        if not self.post_norm or center_feature_scale:
            self.norm = _build_norm_layer(channels, 'LN')
        self.downsample = _DownsampleLayer(channels=channels, norm_layer=norm_layer) if downsample else None

    def forward(self, x):
        for blk in self.blocks:
            x = blk(x)
        if not self.post_norm or self.center_feature_scale:
            x = self.norm(x)
        if self.downsample is not None:
            x = self.downsample(x)
        return x



class SharedInputAdapter(nn.Module):
    \"\"\"
    Improved shared adapter. Two-branch: spatial + scalar.
    No 12->3 bottleneck. out_ch defaults to in_ch (12).
    \"\"\"
    def __init__(self, in_ch=12, out_ch=12, scalar_chs=(8,9,10,11)):
        super().__init__()
        self.scalar_chs = list(scalar_chs)
        self.spatial_conv = nn.Conv2d(in_ch, out_ch, 1, bias=False)
        self.spatial_bn   = nn.BatchNorm2d(out_ch)
        self.scalar_proj  = nn.Linear(len(scalar_chs), out_ch, bias=True)
        self.act = nn.GELU()
        self.blend = nn.Parameter(torch.tensor(0.5))
    def forward(self, x):
        B, C, H, W = x.shape
        s  = self.act(self.spatial_bn(self.spatial_conv(x)))
        sc = x[:, self.scalar_chs, :, :].mean(dim=(-1,-2))
        sc = self.act(self.scalar_proj(sc)).unsqueeze(-1).unsqueeze(-1).expand_as(s)
        w = torch.sigmoid(self.blend)
        return w * s + (1 - w) * sc


class VMambaFeatureExtractor(nn.Module):
    """VMamba backbone, returns feature vector (B, num_features=384) without classifier head."""
    def __init__(self, vssm):
        super().__init__()
        self.vssm = vssm
        self.feature_dim = vssm.num_features  # 48 * 8 = 384

    def forward(self, x):  # (B, 3, H, W)
        x = self.vssm.patch_embed(x)
        if self.vssm.pos_embed is not None:
            pos = self.vssm.pos_embed.permute(0,2,3,1) if not self.vssm.channel_first else self.vssm.pos_embed
            x = x + pos
        for layer in self.vssm.layers:
            x = layer(x)
        # classifier[:-1] = LayerNorm + Permute + AdaptiveAvgPool2d + Flatten (no Linear)
        x = self.vssm.classifier[:-1](x)
        return x  # (B, 384)


class MicroStemLayer(nn.Module):
    def __init__(self, in_ch=3, out_ch=64):
        super().__init__()
        self.conv = nn.Conv2d(in_ch, out_ch, 3, 1, 1, bias=False)
        self.norm = nn.BatchNorm2d(out_ch)
        self.act  = nn.GELU()
    def forward(self, x):
        return self.act(self.norm(self.conv(x))).permute(0, 2, 3, 1)


class InternImageFeatureExtractor(nn.Module):
    """InternImage backbone, returns (B, head_dim=768) feature vector. Probes dim at init."""
    def __init__(self, patch_embed, pos_drop, levels, conv_head, avgpool):
        super().__init__()
        self.patch_embed = patch_embed
        self.pos_drop    = pos_drop
        self.levels      = levels
        self.conv_head   = conv_head
        self.avgpool     = avgpool
        # Probe feature_dim
        self.eval()
        with torch.no_grad():
            _x = torch.zeros(1, 3, 7, 7)
            _x = self.patch_embed(_x); _x = self.pos_drop(_x)
            for lvl in self.levels: _x = lvl(_x)
            _x = _x.permute(0,3,1,2); _x = self.conv_head(_x); _x = self.avgpool(_x)
            self.feature_dim = int(_x.flatten(1).shape[1])
        self.train()

    def forward(self, x):
        x = self.patch_embed(x); x = self.pos_drop(x)
        for lvl in self.levels: x = lvl(x)
        x = x.permute(0,3,1,2); x = self.conv_head(x); x = self.avgpool(x)
        return x.flatten(1)


class HybridInternImageVMambaClassifier(nn.Module):
    """
    Hybrid feature-fusion classifier.
    Architecture:
      Input (B,12,7,7)
        -> SharedInputAdapter -> (B,3,7,7)
        -> VMamba branch      -> (B, 384)   [feature vector, no head]
        -> InternImage branch -> (B, 768)   [feature vector, no head]
        -> Concat             -> (B, 1152)
        -> FusionLayer        -> (B, 576)   Linear+GELU
        -> FusionProjection   -> (B, 576)   Linear+GELU
        -> Dropout(0.3)
        -> Classifier         -> (B, num_classes)
    Fusion dimensions determined programmatically (dry-run probe).
    """
    def __init__(self, num_classes, in_chans=12,
                 vmamba_dims=48, vmamba_depths=None,
                 intern_channels=64, intern_depths=None, intern_groups=None,
                 drop_path_rate=0.1, dropout_rate=0.3,
                 vmamba_forward_type='v05_noz'):
        super().__init__()
        if vmamba_depths is None: vmamba_depths = [2, 2, 4, 2]
        if intern_depths  is None: intern_depths  = [1, 1, 1, 1]
        if intern_groups  is None: intern_groups  = [1, 2, 4, 8]

        _adapter_out = in_chans  # 12->12, no bottleneck
        self.adapter = SharedInputAdapter(in_chans, _adapter_out)

        # VMamba branch
        _vssm = VSSM(
            patch_size=1, in_chans=_adapter_out, num_classes=num_classes,
            depths=vmamba_depths, dims=vmamba_dims, drop_path_rate=drop_path_rate,
            ssm_d_state=16, ssm_ratio=2.0, ssm_dt_rank='auto',
            ssm_act_layer='silu', ssm_conv=3, ssm_conv_bias=True,
            ssm_drop_rate=0.0, ssm_init='v0', forward_type=vmamba_forward_type,
            mlp_ratio=0.0, patch_norm=True, norm_layer='ln',
            downsample_version='v3', patchembed_version='v1',
            use_checkpoint=False, posembed=False, imgsize=7,
        )
        self.vmamba_extractor = VMambaFeatureExtractor(_vssm)

        # InternImage branch
        num_levels   = len(intern_depths)
        num_features = int(intern_channels * 2**(num_levels-1))
        head_dim     = int(num_features * 1.5)
        dpr = [x.item() for x in torch.linspace(0, drop_path_rate, sum(intern_depths))]
        _patch_embed = MicroStemLayer(_adapter_out, intern_channels)
        _pos_drop    = nn.Dropout(p=0.0)
        _levels = nn.ModuleList()
        for i in range(num_levels):
            _levels.append(_InternImageBlock(
                core_op=DCNv3_pytorch, channels=int(intern_channels*2**i),
                depth=intern_depths[i], groups=intern_groups[i], mlp_ratio=4.0, drop=0.0,
                drop_path=dpr[sum(intern_depths[:i]):sum(intern_depths[:i+1])],
                act_layer='GELU', norm_layer='LN', post_norm=False, downsample=(i<num_levels-1),
                layer_scale=None, offset_scale=1.0, with_cp=False,
            ))
        _conv_head = nn.Sequential(nn.Conv2d(num_features, head_dim, 1, bias=False), nn.BatchNorm2d(head_dim), nn.GELU())
        _avgpool   = nn.AdaptiveAvgPool2d((1, 1))
        self.intern_extractor = InternImageFeatureExtractor(_patch_embed, _pos_drop, _levels, _conv_head, _avgpool)

        # Programmatic fusion dimensions
        vm_dim  = self.vmamba_extractor.feature_dim   # 384
        im_dim  = self.intern_extractor.feature_dim   # 768
        cat_dim = vm_dim + im_dim                      # 1152
        fus_dim = cat_dim // 2                         # 576
        self.vmamba_feature_dim = vm_dim
        self.intern_feature_dim = im_dim
        self.concat_dim = cat_dim
        self.fusion_dim = fus_dim

        self.fusion_layer = nn.Sequential(nn.Linear(cat_dim, fus_dim), nn.GELU())
        self.fusion_proj  = nn.Sequential(nn.Linear(fus_dim, fus_dim), nn.GELU())
        self.dropout      = nn.Dropout(p=dropout_rate)
        self.classifier   = nn.Linear(fus_dim, num_classes)

        # Init fusion weights
        for m in [self.fusion_layer, self.fusion_proj, self.classifier]:
            for layer in (m if isinstance(m, nn.Sequential) else [m]):
                if isinstance(layer, nn.Linear):
                    trunc_normal_(layer.weight, std=0.02)
                    if layer.bias is not None: nn.init.constant_(layer.bias, 0)

    def forward(self, x):
        x3   = self.adapter(x)             # (B,3,7,7)
        f_im = self.intern_extractor(x3)   # (B,768)
        f_vm = self.vmamba_extractor(x3)   # (B,384)
        f    = torch.cat([f_im, f_vm], 1)  # (B,1152)
        f    = self.fusion_layer(f)         # (B,576)
        f    = self.fusion_proj(f)          # (B,576)
        f    = self.dropout(f)
        return self.classifier(f)           # (B,num_classes)


# Instantiate
model = HybridInternImageVMambaClassifier(
    num_classes=NUM_CLASSES, vmamba_forward_type=VMAMBA_FORWARD_TYPE,
    dropout_rate=DROPOUT_RATE
).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f"HybridInternImageVMambaClassifier | Classes={NUM_CLASSES} | Params={train_params:,}")
print(f"  VMamba feature dim:    {model.vmamba_feature_dim}")
print(f"  InternImage feat dim:  {model.intern_feature_dim}")
print(f"  Concatenated dim:      {model.concat_dim}")
print(f"  Fusion dim:            {model.fusion_dim}")

try:
    dummy = torch.zeros(2, 12, 7, 7).to(DEVICE)
    with torch.no_grad(): out = model(dummy)
    print(f"Shape test PASSED: input={dummy.shape} -> output={out.shape}")
except Exception as e:
    print(f"Shape test FAILED: {e}")

# Optimizer: no weight decay for SSM params AND DCNv3 bias/offset/mask
def is_no_wd(name):
    return 'A_log' in name or name.endswith('.D') or 'bias' in name or 'offset' in name or 'mask' in name
no_wd = [p for n, p in model.named_parameters() if p.requires_grad and is_no_wd(n)]
wd    = [p for n, p in model.named_parameters() if p.requires_grad and not is_no_wd(n)]
optimizer = torch.optim.Adam(
    [{'params': wd,    'weight_decay': WEIGHT_DECAY},
     {'params': no_wd, 'weight_decay': 0.0}],
    lr=LR, betas=(0.9, 0.999), eps=1e-8
)
print(f"Optimizer: Adam | lr={LR} | wd={WEIGHT_DECAY} | no-wd={len(no_wd)} | wd={len(wd)}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 5. TRAINING LOOP WITH ADAM
# Improvements vs baseline:
#   A) ReduceLROnPlateau(factor=0.5, patience=8, min_lr=1e-6)
#   B) Early stopping (patience=15, monitored: val_loss)
#   C) Mixed-precision AMP when USE_AMP=True and CUDA available
#   D) cuDNN benchmark mode; LR logged every epoch
# ─────────────────────────────────────────────────────────────
RESULTS_DIR = NB_DIR / 'Results' / 'Rabi/Hybrid'
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
CKPT_PATH = RESULTS_DIR / 'best_model.pth'

# Class-weighted CrossEntropyLoss
train_label_ids = np.array([class_to_id[labels[i]] for i in train_idx])
cw_np = compute_class_weight('balanced', classes=np.arange(NUM_CLASSES), y=train_label_ids)
criterion = nn.CrossEntropyLoss(weight=torch.tensor(cw_np, dtype=torch.float32).to(DEVICE))
print(f"Loss: CrossEntropyLoss with balanced class weights")

# cuDNN benchmark: fixed input size => auto-tunes convolution kernels
if DEVICE.type == 'cuda':
    torch.backends.cudnn.benchmark = True

# LR Scheduler: ReduceLROnPlateau on val_loss
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer, mode='min', factor=SCHED_FACTOR, patience=SCHED_PATIENCE,
    min_lr=SCHED_MIN_LR, verbose=False
)
print(f"Scheduler: ReduceLROnPlateau | factor={SCHED_FACTOR} | patience={SCHED_PATIENCE} | min_lr={SCHED_MIN_LR}")

# AMP scaler (no-op on CPU or when USE_AMP=False)
_use_amp = USE_AMP and (DEVICE.type == 'cuda')
scaler = torch.cuda.amp.GradScaler(enabled=_use_amp)
print(f"AMP mixed-precision: {_use_amp}")

# Early stopping state
_es_counter = 0
_es_best_loss = float('inf')
_stopped_early = False

# Training state
history = {'train_loss': [], 'val_loss': [], 'train_acc': [], 'val_acc': [], 'lr': []}
best_val_acc = 0.0
training_start = time.time()

EPOCHS = SMOKE_EPOCHS if SMOKE_TEST else MAX_EPOCHS
print(f"Training for {EPOCHS} epoch(s) | SMOKE_TEST={SMOKE_TEST} | Device={DEVICE}")

for epoch in range(1, EPOCHS + 1):
    # ── Train ────────────────────────────────────────────────
    model.train()
    ep_loss, ep_correct, ep_total = 0.0, 0, 0
    for batch_idx, (inputs, targets) in enumerate(train_loader):
        if SMOKE_TEST and batch_idx >= SMOKE_BATCHES:
            break
        inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
        optimizer.zero_grad()
        with torch.cuda.amp.autocast(enabled=_use_amp):
            outputs = model(inputs)
            loss = criterion(outputs, targets)
        scaler.scale(loss).backward()
        if GRAD_CLIP > 0:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        scaler.step(optimizer)
        scaler.update()
        ep_loss    += loss.item() * inputs.size(0)
        ep_correct += (outputs.argmax(1) == targets).sum().item()
        ep_total   += inputs.size(0)
    tr_loss = ep_loss / ep_total
    tr_acc  = ep_correct / ep_total

    # ── Validate ─────────────────────────────────────────────
    model.eval()
    va_loss, va_correct, va_total = 0.0, 0, 0
    with torch.no_grad():
        for batch_idx, (inputs, targets) in enumerate(val_loader):
            if SMOKE_TEST and batch_idx >= SMOKE_BATCHES:
                break
            inputs, targets = inputs.to(DEVICE), targets.to(DEVICE)
            with torch.cuda.amp.autocast(enabled=_use_amp):
                outputs = model(inputs)
                loss = criterion(outputs, targets)
            va_loss    += loss.item() * inputs.size(0)
            va_correct += (outputs.argmax(1) == targets).sum().item()
            va_total   += inputs.size(0)
    val_loss = va_loss / va_total
    val_acc  = va_correct / va_total

    # ── LR Warmup + Scheduler step ───────────────────────────
    if not SMOKE_TEST:
        if epoch <= LR_WARMUP_EPOCHS:
            warmup_lr = LR * (epoch / LR_WARMUP_EPOCHS)
            for pg in optimizer.param_groups:
                pg['lr'] = warmup_lr
        else:
            scheduler.step(val_loss)
    current_lr = optimizer.param_groups[0]['lr']

    history['train_loss'].append(tr_loss)
    history['val_loss'].append(val_loss)
    history['train_acc'].append(tr_acc)
    history['val_acc'].append(val_acc)
    history['lr'].append(current_lr)

    # ── Save best checkpoint by val_acc ──────────────────────
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
            'val_loss': val_loss,
            'scheduler_state_dict': scheduler.state_dict(),
            'classes': classes,
            'class_to_id': class_to_id,
            'norm_mean': norm_mean.tolist(),
            'norm_std': norm_std.tolist(),
        }, CKPT_PATH)

    # ── Early Stopping (monitored: val_loss) ──────────────────
    if not SMOKE_TEST:
        if val_loss < _es_best_loss - 1e-6:
            _es_best_loss = val_loss
            _es_counter = 0
        else:
            _es_counter += 1
        if _es_counter >= ES_PATIENCE:
            elapsed = time.time() - training_start
            print(f"Epoch {epoch:3d}/{EPOCHS} | "
                  f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | "
                  f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} | "
                  f"Best Val Acc={best_val_acc:.4f} | LR={current_lr:.2e} | {elapsed:.0f}s")
            print(f"
Early stopping at epoch {epoch} (no val_loss improvement for {ES_PATIENCE} epochs)")
            _stopped_early = True
            break

    elapsed = time.time() - training_start
    print(f"Epoch {epoch:3d}/{EPOCHS} | "
          f"Train Loss={tr_loss:.4f} Acc={tr_acc:.4f} | "
          f"Val Loss={val_loss:.4f} Acc={val_acc:.4f} | "
          f"Best Val Acc={best_val_acc:.4f} | LR={current_lr:.2e} | {elapsed:.0f}s")

print(f"\nTraining complete. Best val_acc={best_val_acc:.4f} | Stopped early: {_stopped_early}")
print(f"Checkpoint saved to: {CKPT_PATH}")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 6. EVALUATION ON TEST SET
# Loads best checkpoint selected by validation accuracy.
# Test metrics are NOT used for model selection.
# ─────────────────────────────────────────────────────────────
if not CKPT_PATH.exists():
    print("WARNING: No checkpoint found (smoke test may not have saved). Evaluating current weights.")
else:
    ckpt = torch.load(CKPT_PATH, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    print(f"Loaded best checkpoint: epoch={ckpt['epoch']}, val_acc={ckpt['val_acc']:.4f}")

model.eval()
all_preds, all_targets = [], []
with torch.no_grad():
    for batch_idx, (inputs, targets) in enumerate(test_loader):
        if SMOKE_TEST and batch_idx >= SMOKE_BATCHES:
            break
        inputs = inputs.to(DEVICE)
        outputs = model(inputs)
        preds = outputs.argmax(1).cpu().numpy()
        all_preds.extend(preds)
        all_targets.extend(targets.numpy())

all_preds   = np.array(all_preds)
all_targets = np.array(all_targets)

test_acc      = accuracy_score(all_targets, all_preds)
test_prec_mac = precision_score(all_targets, all_preds, average='macro', zero_division=0)
test_rec_mac  = recall_score(all_targets, all_preds, average='macro', zero_division=0)
test_f1_mac   = f1_score(all_targets, all_preds, average='macro', zero_division=0)
test_f1_wtd   = f1_score(all_targets, all_preds, average='weighted', zero_division=0)

print(f"\nTest Results {'(SMOKE_TEST — limited batches)' if SMOKE_TEST else ''}:")
print(f"  Accuracy:        {test_acc:.4f}")
print(f"  Precision (mac): {test_prec_mac:.4f}")
print(f"  Recall (mac):    {test_rec_mac:.4f}")
print(f"  F1 (macro):      {test_f1_mac:.4f}")
print(f"  F1 (weighted):   {test_f1_wtd:.4f}")
print()
print(classification_report(all_targets, all_preds, target_names=classes, zero_division=0))

In [ ]:
# ─────────────────────────────────────────────────────────────
# 7. RESULT PLOTS
# ─────────────────────────────────────────────────────────────
SMOKE_TAG = ' [SMOKE TEST]' if SMOKE_TEST else ''
epochs_x = range(1, len(history['train_loss'])+1)

# Loss curves
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(epochs_x, history['train_loss'], 'b-', label='Train Loss')
axes[0].plot(epochs_x, history['val_loss'], 'r-', label='Val Loss')
axes[0].set(title=f'Loss Curves{SMOKE_TAG}', xlabel='Epoch', ylabel='Loss')
axes[0].legend(); axes[0].grid(True)
axes[1].plot(epochs_x, history['train_acc'], 'b-', label='Train Acc')
axes[1].plot(epochs_x, history['val_acc'], 'r-', label='Val Acc')
axes[1].set(title=f'Accuracy Curves{SMOKE_TAG}', xlabel='Epoch', ylabel='Accuracy')
axes[1].legend(); axes[1].grid(True)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'training_curves.png', dpi=120, bbox_inches='tight')
plt.show(); print(f"Saved: training_curves.png")

# Confusion matrix
cm = confusion_matrix(all_targets, all_preds)
fig, ax = plt.subplots(figsize=(max(8, NUM_CLASSES), max(6, NUM_CLASSES)))
sns.heatmap(cm, annot=(NUM_CLASSES <= 20), fmt='d', cmap='Blues',
            xticklabels=classes, yticklabels=classes, ax=ax, linewidths=0.3)
ax.set(title=f'Confusion Matrix{SMOKE_TAG}', xlabel='Predicted', ylabel='True')
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'confusion_matrix.png', dpi=120, bbox_inches='tight')
plt.show(); print(f"Saved: confusion_matrix.png")

# Per-class accuracy
per_class_acc = []
for i in range(NUM_CLASSES):
    mask = all_targets == i
    if mask.sum() > 0:
        per_class_acc.append(float((all_preds[mask] == all_targets[mask]).mean()))
    else:
        per_class_acc.append(0.0)
fig, ax = plt.subplots(figsize=(max(10, NUM_CLASSES*0.7), 5))
bars = ax.bar(classes, per_class_acc, color='steelblue', edgecolor='black', linewidth=0.5)
ax.set(title=f'Per-Class Accuracy{SMOKE_TAG}', ylabel='Accuracy', ylim=(0, 1.05))
ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=8)
for bar, acc in zip(bars, per_class_acc):
    ax.text(bar.get_x() + bar.get_width()/2., bar.get_height() + 0.01, f'{acc:.2f}', ha='center', va='bottom', fontsize=6)
plt.tight_layout()
plt.savefig(RESULTS_DIR / 'per_class_accuracy.png', dpi=120, bbox_inches='tight')
plt.show(); print(f"Saved: per_class_accuracy.png")

# Learning rate curve
if 'lr' in history and len(history['lr']) > 0:
    fig, ax = plt.subplots(figsize=(8, 3))
    ax.semilogy(range(1, len(history['lr'])+1), history['lr'], 'g-', linewidth=1.5)
    ax.set(title=f'Learning Rate{SMOKE_TAG}', xlabel='Epoch', ylabel='LR (log scale)')
    ax.grid(True)
    plt.tight_layout()
    plt.savefig(RESULTS_DIR / 'lr_curve.png', dpi=120, bbox_inches='tight')
    plt.show(); print(f"Saved: lr_curve.png")

In [ ]:
# ─────────────────────────────────────────────────────────────
# 8. FINAL EXPERIMENT REPORT
# ─────────────────────────────────────────────────────────────
total_params = sum(p.numel() for p in model.parameters())
train_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
duration = time.time() - training_start
gpu_mem = f"{torch.cuda.max_memory_allocated()/1e9:.2f} GB" if torch.cuda.is_available() else "N/A (CPU)"

print("=" * 65)
print(f"  EXPERIMENT: Hybrid InternImage+VMamba + Rabi + Adam")
print("=" * 65)
print(f"  Dataset:          {PICKLE_PATH.name}")
print(f"  Classes:          {NUM_CLASSES}")
print(f"  Train / Val / Test: {len(train_idx):,} / {len(val_idx):,} / {len(test_idx):,}")
print(f"  Batch size:       {BS}")
print(f"  Epochs trained:   {len(history['train_loss'])}")
print(f"  Best val acc:     {best_val_acc:.4f}")
print(f"  Test accuracy:    {test_acc:.4f}")
print(f"  Test F1 (macro):  {test_f1_mac:.4f}")
print(f"  Test F1 (wtd):    {test_f1_wtd:.4f}")
print(f"  Total params:     {total_params:,}")
print(f"  Trainable params: {train_params:,}")
print(f"  Device:           {DEVICE}")
print(f"  Peak GPU memory:  {gpu_mem}")
print(f"  Training duration: {duration/60:.1f} min")
print(f"  Checkpoint:       {CKPT_PATH}")
print(f"  Results dir:      {RESULTS_DIR}")
print(f"  SMOKE_TEST:       {SMOKE_TEST}")
print("=" * 65)
if SMOKE_TEST:
    print()
    print("  *** SMOKE TEST COMPLETE — set SMOKE_TEST=False for full training ***")
    print("  *** On GPU: set FORWARD_TYPE='v2' (VMamba) and compile DCNv3      ***")